# CDSE TS Browser

This notebook serves to construct a geojson file to use in https://github.com/jonasViehweger/cdse-tsbrowser/

It also constructs direct links to single samples.

In [89]:
import json
import urllib

import disfor
import geopandas as gpd
import polars as pl

In [90]:
samples = gpd.read_parquet(disfor.get("samples.parquet"))
samples.fillna("", inplace=True)
labels = pl.read_parquet(disfor.get("labels.parquet")).with_columns(
    is_event=pl.col.start.dt.day() == pl.col.end.dt.day()
)
with disfor.get("classes.json").open() as f:
    classes_mapping = json.load(f)

In [54]:
classes_mapping

{'100': 'Healthy Vegetation',
 '110': 'Undisturbed Forest',
 '120': 'Revegetation',
 '121': 'With Trees (after clear cut)',
 '122': 'Canopy closing (after thinning/defoliation)',
 '123': 'Without Trees (shrubs and grasses, no reforestation visible)',
 '200': 'Disturbed',
 '210': 'Planned',
 '211': 'Clear Cut',
 '212': 'Thinning',
 '213': 'Forestry Mulching (Non Forest Vegetation Removal)',
 '220': 'Salvage',
 '221': 'After Biotic Disturbance',
 '222': 'After Abiotic Disturbance',
 '230': 'Biotic',
 '231': 'Bark Beetle (with decline)',
 '232': 'Gypsy Moth (temporary)',
 '240': 'Abiotic',
 '241': 'Drought',
 '242': 'Wildfire',
 '243': 'Wind',
 '244': 'Avalanche',
 '245': 'Flood'}

In [ ]:
campaign_setup = {
    "type": "FeatureCollection",
    "campaign": {
        "name": "DISFOR",
        "startDate": "2015-01-01",
        "endDate": "2025-01-01",
        "flagLabels": classes_mapping,
        "fields": [
            {
                "key": "sample_id",
                "label": "Sample ID",
                "type": "display",
                "required": True,
                "session_persistent": False,
            },
            {
                "key": "source",
                "label": "Source",
                "type": "display",
                "required": True,
                "session_persistent": False,
            },
            {
                "key": "cluster_id",
                "label": "Cluster ID",
                "type": "display",
                "required": True,
                "session_persistent": False,
            },
            {
                "key": "confidence",
                "label": "Confidence",
                "type": "select",
                "options": ["high", "medium", "low"],
                "required": True,
                "session_persistent": False,
            },
            {
                "key": "comment",
                "label": "Comment",
                "type": "text",
                "required": False,
                "session_persistent": False,
            },
            {
                "key": "interpreter",
                "label": "Interpreter",
                "type": "text",
                "required": True,
                "session_persistent": True,
            },
        ],
    },
    "features": [],
}

In [56]:
# single sample schema:
feature_example = {
    "type": "Feature",
    "geometry": {"type": "Point", "coordinates": ["lon", "lat"]},
    "properties": {
        "sample_id": None,
        "flags": {"2020-07-13": "1", "2021-08-02": "2"},
        "confidence": "High",
        "comment": "Clear bark beetle signal, recovery visible by 2021",
        "interpreter": "jdoe",
    },
}

In [57]:
samples

,sample_id,original_sample_id,interpreter,dataset,source,source_description,s2_tile,cluster_id,cluster_description,comment,confidence,geometry
0,0,0,vij,1,EFFIS,"Evoland Project, EFFIS Source of Wildfire Poly...",30SUF,0.0,Damage polygons,"border, thinning, then clear cut",high,POINT (-4.12212 36.74179)
1,1,1,vij,1,EFFIS,"Evoland Project, EFFIS Source of Wildfire Poly...",30SUF,1.0,Damage polygons,"unclear progression, edge",medium,POINT (-4.12161 36.74231)
2,2,2,vij,1,EFFIS,"Evoland Project, EFFIS Source of Wildfire Poly...",30SUF,2.0,Damage polygons,plantation,high,POINT (-4.1192 36.74203)
3,3,3,vij,1,EFFIS,"Evoland Project, EFFIS Source of Wildfire Poly...",30SUF,3.0,Damage polygons,plantation,high,POINT (-4.12845 36.75831)
4,4,4,vij,1,EFFIS,"Evoland Project, EFFIS Source of Wildfire Poly...",30SUF,5.0,Damage polygons,clear cut,high,POINT (-4.12816 36.75908)
...,...,...,...,...,...,...,...,...,...,...,...,...
3823,3818,16066,vij,3,FORWIND + Copernicus Emergency Service,https://mapping.emergency.copernicus.eu/activa...,,SI20200205,"Id of the Event, given as ISO2 + Date of storm",,high,POINT (14.39175 46.2299)
3824,3819,16067,vij,3,FORWIND + Copernicus Emergency Service,https://mapping.emergency.copernicus.eu/activa...,,SI20200205,"Id of the Event, given as ISO2 + Date of storm",unclear salvage,high,POINT (14.4004 46.2291)
3825,3820,16070,vij,3,FORWIND + Copernicus Emergency Service,https://mapping.emergency.copernicus.eu/activa...,,SI20200205,"Id of the Event, given as ISO2 + Date of storm",,high,POINT (14.39046 46.24575)
3826,3821,16071,vij,3,FORWIND + Copernicus Emergency Service,https://mapping.emergency.copernicus.eu/activa...,,SI20200205,"Id of the Event, given as ISO2 + Date of storm",unclear salvage,high,POINT (14.39247 46.2441)


In [58]:
events = labels.filter("is_event").select(
    "sample_id", "label", pl.col.start.alias("flag_date")
)
segments = (
    labels.filter(~pl.col.is_event)
    .unpivot(["start", "end"], index=["sample_id", "label"], value_name="flag_date")
    .drop("variable")
)
all_labels = pl.concat([events, segments])

In [59]:
all_labels

sample_id,label,flag_date
u16,u16,"datetime[ms, UTC]"
0,212,2022-04-08 00:00:00 UTC
0,211,2024-04-02 00:00:00 UTC
1,211,2023-06-22 00:00:00 UTC
2,212,2022-04-08 00:00:00 UTC
3,212,2022-05-08 00:00:00 UTC
…,…,…
3820,120,2024-12-30 00:00:00 UTC
3821,110,2020-01-21 00:00:00 UTC
3821,121,2024-12-30 00:00:00 UTC


In [60]:
flags_tuple = (
    all_labels.filter(sample_id=0)
    .sort("flag_date")
    .select(pl.col.flag_date.dt.strftime("%Y-%m-%d"), "label")
    .rows_by_key("flag_date", unique=True)
)
flags = {k: str(v[0]) for k, v in flags_tuple.items()}

In [61]:
features = []
for row, f in samples.iterrows():
    sample_id = f["sample_id"]
    flags_tuple = (
        all_labels.filter(sample_id=sample_id)
        .sort("flag_date")
        .select(pl.col.flag_date.dt.strftime("%Y-%m-%d"), "label")
        .rows_by_key("flag_date", unique=True)
    )
    flags = {k: str(v[0]) for k, v in flags_tuple.items()}
    feature = {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [f["geometry"].x, f["geometry"].y],
        },
        "properties": {
            "sample_id": sample_id,
            "flags": flags,
            "confidence": f["confidence"],
            "cluster_id": f["cluster_id"],
            "source": f["source"],
            "comment": f["comment"],
            "interpreter": f["interpreter"],
        },
    }
    features.append(feature)

In [62]:
features

[{'type': 'Feature',
  'geometry': {'type': 'Point',
   'coordinates': [-4.122119320177979, 36.741785367381176]},
  'properties': {'sample_id': 0,
   'flags': {'2016-11-10': '110',
    '2022-03-09': '110',
    '2022-04-08': '212',
    '2024-04-02': '211'},
   'confidence': 'high',
   'cluster_id': '0.0',
   'source': 'EFFIS',
   'comment': 'border, thinning, then clear cut',
   'interpreter': 'vij'}},
 {'type': 'Feature',
  'geometry': {'type': 'Point',
   'coordinates': [-4.121614665341534, 36.74231320914648]},
  'properties': {'sample_id': 1,
   'flags': {'2016-11-10': '110', '2023-05-03': '110', '2023-06-22': '211'},
   'confidence': 'medium',
   'cluster_id': '1.0',
   'source': 'EFFIS',
   'comment': 'unclear progression, edge',
   'interpreter': 'vij'}},
 {'type': 'Feature',
  'geometry': {'type': 'Point',
   'coordinates': [-4.1191960120097075, 36.7420339745919]},
  'properties': {'sample_id': 2,
   'flags': {'2016-11-10': '110', '2022-04-01': '110', '2022-04-08': '212'},
   'co

In [63]:
campaign_setup["features"] = features

In [64]:
with open("test.geojson", "w") as fp:
    json.dump(campaign_setup, fp, indent=4)

In [65]:
with open("test.geojson", "r") as fp:
    campaign_setup = json.load(fp)

In [66]:
campaign_setup

{'type': 'FeatureCollection',
 'campaign': {'name': 'DISFOR',
  'startDate': '2015-01-01',
  'endDate': '2025-01-31',
  'flagLabels': {'100': 'Healthy Vegetation',
   '110': 'Undisturbed Forest',
   '120': 'Revegetation',
   '121': 'With Trees (after clear cut)',
   '122': 'Canopy closing (after thinning/defoliation)',
   '123': 'Without Trees (shrubs and grasses, no reforestation visible)',
   '200': 'Disturbed',
   '210': 'Planned',
   '211': 'Clear Cut',
   '212': 'Thinning',
   '213': 'Forestry Mulching (Non Forest Vegetation Removal)',
   '220': 'Salvage',
   '221': 'After Biotic Disturbance',
   '222': 'After Abiotic Disturbance',
   '230': 'Biotic',
   '231': 'Bark Beetle (with decline)',
   '232': 'Gypsy Moth (temporary)',
   '240': 'Abiotic',
   '241': 'Drought',
   '242': 'Wildfire',
   '243': 'Wind',
   '244': 'Avalanche',
   '245': 'Flood'},
  'fields': [{'key': 'sample_id',
    'label': 'Sample ID',
    'type': 'display',
    'required': True,
    'session_persistent': Fal

In [67]:
campaign_schema = campaign_setup["campaign"]
campaign_schema

{'name': 'DISFOR',
 'startDate': '2015-01-01',
 'endDate': '2025-01-31',
 'flagLabels': {'100': 'Healthy Vegetation',
  '110': 'Undisturbed Forest',
  '120': 'Revegetation',
  '121': 'With Trees (after clear cut)',
  '122': 'Canopy closing (after thinning/defoliation)',
  '123': 'Without Trees (shrubs and grasses, no reforestation visible)',
  '200': 'Disturbed',
  '210': 'Planned',
  '211': 'Clear Cut',
  '212': 'Thinning',
  '213': 'Forestry Mulching (Non Forest Vegetation Removal)',
  '220': 'Salvage',
  '221': 'After Biotic Disturbance',
  '222': 'After Abiotic Disturbance',
  '230': 'Biotic',
  '231': 'Bark Beetle (with decline)',
  '232': 'Gypsy Moth (temporary)',
  '240': 'Abiotic',
  '241': 'Drought',
  '242': 'Wildfire',
  '243': 'Wind',
  '244': 'Avalanche',
  '245': 'Flood'},
 'fields': [{'key': 'sample_id',
   'label': 'Sample ID',
   'type': 'display',
   'required': True,
   'session_persistent': False},
  {'key': 'source',
   'label': 'Source',
   'type': 'display',
   '

In [68]:
# convert features in campaign to urls:

base_url = "https://cdse-tsbrowser.jonasvieh.workers.dev/?"

In [69]:
query_params = {}

In [70]:
query_params["start"] = campaign_schema.pop("startDate")
query_params["end"] = campaign_schema.pop("endDate")
campaign_schema["campaign"] = campaign_schema.pop("name")

In [71]:
query_params["schema"] = json.dumps(campaign_schema, separators=(",", ":"))

In [ ]:
urls = []
for feature in campaign_setup["features"]:
    query_params["lon"] = feature["geometry"]["coordinates"][0]
    query_params["lat"] = feature["geometry"]["coordinates"][1]
    query_params["sample"] = json.dumps(feature["properties"], separators=(",", ":"))
    urls.append(
        {
            "sample_id": feature["properties"]["sample_id"],
            "url": base_url + urllib.parse.urlencode(query_params),
        }
    )

In [91]:
pl.DataFrame(urls)

sample_id,url
i64,str
0,"""https://cdse-tsbrowser.jonasvi…"
1,"""https://cdse-tsbrowser.jonasvi…"
2,"""https://cdse-tsbrowser.jonasvi…"
3,"""https://cdse-tsbrowser.jonasvi…"
4,"""https://cdse-tsbrowser.jonasvi…"
…,…
3818,"""https://cdse-tsbrowser.jonasvi…"
3819,"""https://cdse-tsbrowser.jonasvi…"
3820,"""https://cdse-tsbrowser.jonasvi…"
